#### beads-id: yaralpho-b5i
#### beads-status: synced

# $TITLE$: Add possibility to see events live as they come while a run is in progress

## Context

we expose /app endpoint for users so that they can see batch and run progress on ui. here is an example url:

http://localhost:8080/app?batch=batch-1770745955350889188&run=run-1770747214887925290

the logic exists under `internal/app` folder 

## Problem

this view isn't "live" - user needs to refresh the page to see new events. we want to make it live so that as events come in, they are reflected on the page without refresh.

## Solution

User loads the page and fetches events the same way it happens now. 

UI establishes websocket connection to the backend and provides timestamp of the last event it has. 

Backend accepts websocket connection, keep is it alive and streams new events as they come in.

UI listens to the websocket and updates the page with new events as they come in, using rendering logic that is already in place for existing events.

## Supporting documentation

**Existing endpoints (internal/app/routes.go)**
- GET /health
- POST /add
- GET /batches
- GET /batches/{id}
- GET /batches/{id}/progress
- GET /runs (batch_id optional)
- GET /runs/{id}
- GET /runs/{id}/events (limit query; caps via defaultEventsLimit/maxEventsLimit in run_events_handler.go); UI served at /app and /app/static

**Storage model (internal/storage/models.go, interfaces.go, mongo/session_events.go)**
- Batch: id, created_at, input_items, status (created/running/idle/done/failed/blocked_auth), optional summary and session_name.
- TaskRun: run_id, batch_id, task_ref, optional epic_ref, session_id, started_at, optional finished_at, status (running/succeeded/failed/stopped), optional result.
- SessionEvent: batch_id, run_id, session_id, event payload (map), ingested_at; stored in Mongo session_events sorted by ingested_at ASC; ListSessionEvents returns all for a session.
- BatchProgress aggregates counts (total/pending/running/succeeded/failed/stopped); storage interface exposes CRUD for batches/runs, InsertSessionEvent, ListSessionEvents, GetBatchProgress.

**Event lifecycle (internal/consumer/task_helpers.go)**
- executeTask starts Copilot session (session_id, events chan) and creates TaskRun (status running).
- For each event from the session channel, InsertSessionEvent persists it with ingested_at=now.
- Loop breaks on session idle, channel close, or context cancel; updates TaskRun status (succeeded/stopped/failed) and finished_at; updates batch status and emits notifier callbacks (NotifyTaskFinished/NotifyBatchIdle/NotifyError).
- Helper functions re-list events for structured output after run completion.

#### beads-id: yaralpho-b5i.1
#### beads-status: synced

# $TITLE$: Task 1: Streaming module scaffolding

Create internal/streaming module scaffold.
- Add folder internal/streaming.
- Add AGENTS.md describing purpose: session event streaming/bus for WS live updates.
Acceptance: folder exists, AGENTS.md present with brief purpose.

#### beads-id: yaralpho-b5i.2
#### beads-status: synced

# $TITLE$: Task 2: Streaming interfaces

Define interfaces under internal/streaming.
- File: internal/streaming/interfaces.go (or similar).
- Interfaces: e.g., AgentSessionEventBus with Publish(ctx, sessionID, event) error; Subscribe(ctx, sessionID) (chan storage.SessionEvent, func()).
- Keep types aligned with storage.SessionEvent; consider cursor/ordering contract.
Acceptance: interfaces defined, code compiles.

#### beads-id: yaralpho-b5i.3
#### beads-status: synced

# $TITLE$: Task 3: Bus contracts and subscribe lifecycle

Define bus contracts and implement subscription lifecycle with cleanup.

Scope
- Add Bus interface (Publish(ctx, sessionID string, evt storage.SessionEvent) error; Subscribe(ctx, sessionID string) (Subscription, error)).
- Add Subscription type: Events <-chan storage.SessionEvent, Done() <-chan struct{}, Close() error (idempotent, cleans up map entry and channel).
- Define SlowConsumerPolicy enum + Config (bufferSize default ~64; optional policy selection; optional logger hook) in bus.go.
- Implement memoryBus struct shape (subs map[string]map[*subscriber]struct{}, mutex, cfg). Implement Subscribe: create buffered chan, register under session, return Subscription. Implement Close + ctx watcher: remove from map, close chan safely, delete session bucket when empty.

Tests (memory_bus_test.go)
- Subscribe registers a subscriber in the right session bucket; Events channel is buffered to default size.
- Close is idempotent, removes the subscriber, closes channel, and deletes empty session bucket.
- Context cancel triggers cleanup (watcher removes subscriber and closes channel).
- Isolation: session A/B subscriptions do not share buckets.

Acceptance: interfaces/types compile; tests above pass; no publish logic yet.

#### beads-id: yaralpho-b5i.4
#### beads-status: synced

# $TITLE$: Task 3.1: Publish fan-out and slow-policy handling

Implement bus Publish fan-out per session; enforce buffer size and slow-consumer policy (drop oldest/block/error) with metrics/log hooks.
Acceptance:
- In-order publish fan-out to all subscribers for a session
- Buffer-full triggers configured policy and surfaces via error/metrics
- Concurrent publishes stay isolated across sessions; tests cover policy branches

#### beads-id: yaralpho-b5i.5
#### beads-status: synced

# $TITLE$: Task 3.2: Limits and idle pruning

Add bus limits (max subs per session, max sessions) and idle pruning with timeouts; define behavior when caps exceeded.
Acceptance:
- Idle timeout removes subscriber/bucket and closes channels without leaks
- Exceeding limits follows configured policy (error/drop) with logs/metrics
- Tests cover idle timer cleanup and cap-hit behavior

#### beads-id: yaralpho-b5i.6
#### beads-status: synced

# $TITLE$: Task 4: Consumer publish integration

Wire consumer to stream bus: after InsertSessionEvent, publish to bus with batch/run/session identifiers; handle publish errors deterministically.
Acceptance:
- Consumer constructed with bus dependency; publish invoked for each persisted session event
- Publish errors surfaced via structured log/metric and chosen retry/stop behavior documented
- Integration test with fake bus asserts publish inputs and error path handling

#### beads-id: yaralpho-b5i.7
#### beads-status: synced

# $TITLE$: Task 5: WebSocket event stream handler

Add WS handler (e.g., /runs/{id}/events/live) accepting batch/run ids and optional last_ingested; validate params, upgrade, subscribe to bus, and manage ping/pong/close codes.
Acceptance:
- Handler rejects missing/invalid params with appropriate close codes/messages
- Successful upgrade subscribes to bus for the run and wires cleanup on close/cancel
- Logs include batch/run/session and close reasons; tests cover upgrade success/failure cases

#### beads-id: yaralpho-b5i.8
#### beads-status: synced

# $TITLE$: Task 5.1: last_ingested filtering and envelopes

Implement cursor handling and message envelopes. On connect, apply last_ingested (or 0) to backfill from storage then stream live bus events; envelope payloads with type (event/error/heartbeat) and cursor.
Acceptance:
- Missing/invalid cursor handled gracefully with documented default; future cursor rejected
- Backfill returns only events after cursor and switches to live without duplication or gaps
- Envelope schema documented; tests cover ordering and cursor edge cases

#### beads-id: yaralpho-b5i.9
#### beads-status: synced

# $TITLE$: Task 5.2: heartbeats and write-failure handling

Send heartbeat frames on interval; detect stalled clients and close with reason. Handle WS write errors by logging context and closing subscription cleanly.
Acceptance:
- Heartbeat interval documented; missing pong or stalled write triggers close with code/reason
- Write failures remove subscriber and close WS; bus subscription cleaned up
- Tests simulate heartbeat timeout and write failure paths

#### beads-id: yaralpho-b5i.10
#### beads-status: synced

# $TITLE$: Task 6: UI live events wiring

Wire app UI to live WS stream: open connection after initial fetch using run id + last_ingested, and route incoming events through existing render path without refresh.
Acceptance:
- WS URL built from current batch/run params with last_ingested from latest event
- Incoming events merge into existing view without duplicates; initial static render preserved
- Logs/telemetry capture connect/disconnect and errors

#### beads-id: yaralpho-b5i.11
#### beads-status: synced

# $TITLE$: Task 6.1: WebSocket stream attach and merge

Implement client-side state merge: maintain run events cache keyed by ingest time/id, append in-order, and dedupe. Keep pagination/limit behavior consistent with existing view.
Acceptance:
- Merge reducer handles in-order and out-of-order arrivals without duplication
- Existing rendering components reused; pagination/limits remain unchanged
- Unit test covers merge scenarios and edge cases

#### beads-id: yaralpho-b5i.12
#### beads-status: synced

# $TITLE$: Task 6.2: Reconnect, backoff, and failure UX

Implement reconnect with jittered backoff and max attempts; surface UI status (banner/toast) and fallback to manual refresh on exhaustion.
Acceptance:
- Reconnect policy documented (delays, max attempts) and implemented with jitter
- User sees connection status/errors; manual refresh fallback available after failures
- Tests simulate drop/retry exhaustion and verify UX signals

#### beads-id: yaralpho-b5i.18
#### beads-status: synced

# $TITLE$: Task 7: Sticky run header + scrollable events

Scope\n- UI layout: keep run header block pinned at top while events list scrolls.\n- Create a dedicated scroll container for events (e.g., .events-scroll) without shifting existing render.\n- Ensure header info remains visible during scroll (CSS/sticky), preserving current buttons and status layout.\n- Update/extend JS rendering to target the scroll container without duplicating nodes.\n\nTests\n- Unit test verifies header stays outside scroll container and event list renders inside (DOM structure).\n- Unit test for scroll container height/overflow setup (class/attribute presence).\n\nAcceptance\n- Header stays visible while events scroll.\n- Events append inside scrollable area without layout regressions.\n- Tests pass.


#### beads-id: yaralpho-b5i.19
#### beads-status: synced

# $TITLE$: Task 7.1: Auto-follow vs manual scroll state

Scope\n- Track scroll mode state: following (auto-scroll) vs manual (user scrolled up).\n- If user is at bottom when new events arrive, auto-scroll to keep newest visible.\n- If user scrolls up (or away from bottom threshold), disable auto-follow until user returns to bottom.\n- Expose a small UI cue (e.g., "Live" / "Paused") or class for status styling.\n\nTests\n- Unit test for bottom-threshold detection and auto-scroll behavior.\n- Unit test for toggling to manual mode on user scroll up and resuming on scroll-to-bottom.\n\nAcceptance\n- At bottom: new events keep view pinned to latest.\n- Scrolling up freezes position; new events do not yank view.\n- Tests cover mode transitions and scroll behavior.

#### beads-id: yaralpho-b5i.20
#### beads-status: synced

# $TITLE$: Task 7.2: Runs list shows task ref + event count

Scope\n- Extend runs list response/model to include task_ref and total_events for each run.\n- Update runs list handler and storage query (count events by run) with deterministic ordering and performance note.\n- Update runs list UI table to render Task Ref and Total Events columns.\n\nTests\n- Handler test covers task_ref + total_events fields on list response.\n- UI unit test verifies columns render and values map to response.\n\nAcceptance\n- Runs list shows task ref (e.g., yaralpho-b5i.12) and total events per run.\n- No regressions in existing runs list rendering.\n- Tests pass.